# Module 02: Container Build & Artifact Registry Setup
In this module, you will set up Google Artifact Registry and build Docker container images for CPU, GPU, and TPU JAX workloads using Google Cloud Build.

### Learning Objectives:
1. Create a Docker Artifact Registry repository in GCP.
2. Build the zero-quota **CPU JAX image** (`src/Dockerfile.cpu`).
3. Build the **GPU JAX image** (`src/Dockerfile.gpu`) and **TPU JAX image** (`src/Dockerfile.tpu`).

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
import config
cfg = config.load_config("../config.env")

PROJECT_ID = cfg["PROJECT_ID"]
REGION = cfg["REGION"]
REPO = cfg["ARTIFACT_REGISTRY_REPO"]
CPU_IMAGE = cfg["CPU_IMAGE_NAME"]
GPU_IMAGE = cfg["GPU_IMAGE_NAME"]
TPU_IMAGE = cfg["TPU_IMAGE_NAME"]
TAG = cfg["IMAGE_TAG"]

CPU_FULL_IMAGE = f"{REGION}-docker.pkg.dev/{PROJECT_ID}/{REPO}/{CPU_IMAGE}:{TAG}"
GPU_FULL_IMAGE = f"{REGION}-docker.pkg.dev/{PROJECT_ID}/{REPO}/{GPU_IMAGE}:{TAG}"
TPU_FULL_IMAGE = f"{REGION}-docker.pkg.dev/{PROJECT_ID}/{REPO}/{TPU_IMAGE}:{TAG}"

print(f"CPU Image: {CPU_FULL_IMAGE}")
print(f"GPU Image: {GPU_FULL_IMAGE}")
print(f"TPU Image: {TPU_FULL_IMAGE}")

## 1. Create Google Artifact Registry Repository

In [ ]:
!gcloud artifacts repositories create {REPO} \
    --repository-format=docker \
    --location={REGION} \
    --description="JAX Multi-Node Container Repository"

In [ ]:
!gcloud auth configure-docker {REGION}-docker.pkg.dev --quiet

## 2. Build JAX CPU Container Image (Primary Zero-Quota Image)

In [ ]:
!gcloud builds submit ../src/ \
    --config=- <<EOF
steps:
- name: 'gcr.io/cloud-builders/docker'
  args: ['build', '-t', '{CPU_FULL_IMAGE}', '-f', 'Dockerfile.cpu', '.']
images:
- '{CPU_FULL_IMAGE}'
EOF

## 3. Build JAX GPU & TPU Container Images

In [ ]:
!gcloud builds submit ../src/ \
    --config=- <<EOF
steps:
- name: 'gcr.io/cloud-builders/docker'
  args: ['build', '-t', '{GPU_FULL_IMAGE}', '-f', 'Dockerfile.gpu', '.']
images:
- '{GPU_FULL_IMAGE}'
EOF

In [ ]:
!gcloud builds submit ../src/ \
    --config=- <<EOF
steps:
- name: 'gcr.io/cloud-builders/docker'
  args: ['build', '-t', '{TPU_FULL_IMAGE}', '-f', 'Dockerfile.tpu', '.']
images:
- '{TPU_FULL_IMAGE}'
EOF

## 4. Verify Built Images in Artifact Registry

In [ ]:
!gcloud artifacts docker images list {REGION}-docker.pkg.dev/{PROJECT_ID}/{REPO}